# RapidMatch demo

Build a control group whose pre-campaign features look like the campaign target group.

This notebook:
1. Builds a small synthetic dataset with a known imbalance
2. Runs `ControlMatcher`
3. Inspects coverage, pair quality, and a simple before/after mean check

Run from the repo root so `import rapidmatch` resolves.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "rapidmatch").exists() and (root.parent / "rapidmatch").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd

from rapidmatch import ControlMatcher, MatchConfig

## 1. Synthetic data

200 targets clustered around income 70k / age 42.

800 controls: half from the same distribution (easy matches), half younger and poorer (deliberate mismatch). A naive random control sample would be pulled toward the poor/young mass. Matching should prefer the lookalike half.

In [ ]:
rng = np.random.default_rng(42)
n_target, n_like, n_other = 200, 400, 400

target = pd.DataFrame({
    "id": np.arange(n_target, dtype=int),
    "income": rng.normal(70000, 8000, n_target),
    "age": rng.normal(42, 6, n_target),
    "region": rng.choice(["N", "S", "E", "W"], n_target),
    "is_target": 1,
})
like = pd.DataFrame({
    "id": np.arange(n_target, n_target + n_like, dtype=int),
    "income": rng.normal(70000, 8000, n_like),
    "age": rng.normal(42, 6, n_like),
    "region": rng.choice(["N", "S", "E", "W"], n_like),
    "is_target": 0,
})
other = pd.DataFrame({
    "id": np.arange(n_target + n_like, n_target + n_like + n_other, dtype=int),
    "income": rng.normal(45000, 9000, n_other),
    "age": rng.normal(30, 7, n_other),
    "region": rng.choice(["N", "S", "E", "W"], n_other),
    "is_target": 0,
})
df = pd.concat([target, like, other], ignore_index=True)
df.head()

In [ ]:
df.groupby("is_target")[["income", "age"]].mean().round(1)

## 2. Configure and run

- `match_vars` are used both to stratify and to score distance
- `weights` boost income over age in the distance
- `tolerance=0.2` keeps pairs at or above the 20th percentile of accepted strengths (fairly loose, good for a first look)
- `min_control_pool_size=3` flags thin strata but still matches them

In [ ]:
config = MatchConfig(
    match_vars=["income", "age", "region"],
    treatment_col="is_target",
    id_col="id",
    weights={"income": 1.5, "age": 1.0},
    n=1,
    tolerance=0.2,
    min_control_pool_size=3,
    n_bins=4,
)
matcher = ControlMatcher(config)
result = matcher.fit_match(df)
result.coverage_summary

## 3. Who got a match?

`match_status`:
- `matched` — kept after tolerance
- `below_tolerance` — had a candidate, too weak
- `no_control_available` — empty control cell
- `unmatched` — eligible, lost the control to a stronger pair

`thin_stratum` is a separate flag and can sit on a `matched` row.

In [ ]:
result.targets["match_status"].value_counts()

In [ ]:
matched = result.pairs[result.pairs["match_status"] == "matched"].copy()
matched.sort_values("match_strength", ascending=False).head(10)

## 4. Did matching actually balance income and age?

Compare target means against (a) a random control sample of the same size and (b) the matched controls.

In [ ]:
target_rows = df[df["is_target"] == 1]
control_rows = df[df["is_target"] == 0]
random_ctrl = control_rows.sample(n=len(matched), random_state=0)
matched_ctrl = df.set_index("id").loc[matched["control_id"].astype(int)]

def means(frame, label):
    return pd.Series({
        "income": frame["income"].mean(),
        "age": frame["age"].mean(),
        "n": len(frame),
    }, name=label)

balance = pd.concat([
    means(target_rows, "target"),
    means(random_ctrl, "random_control"),
    means(matched_ctrl, "matched_control"),
], axis=1).T
balance.round(2)

Matched control means should sit much closer to the target than the random sample. Strength should all lie in (0, 1].

In [ ]:
print("n matched pairs:", len(matched))
print("unique controls:", matched["control_id"].nunique())
print("strength min/max:", matched["match_strength"].min(), matched["match_strength"].max())
print("thin among matched:", int(matched["thin_stratum"].sum()))
print("bin edges used:", matcher.bin_edges)
print("tolerance cutoff:", result.cutoff)

## 5. Try it on a CSV

`fit_match` accepts a file path. CSV/Parquet/Feather stream through DuckDB; Excel is loaded then streamed.

In [ ]:
csv_path = root / "demo_input.csv"
df.to_csv(csv_path, index=False)
from_file = ControlMatcher(config).fit_match(str(csv_path))
from_file.coverage_summary

## 6. Knobs worth turning

| knob | effect |
|---|---|
| `n=2` | up to two controls per target (still no reuse) |
| `tolerance=0.8` | keep only the strongest 20% of pairs |
| `n_bins=6` | finer strata, more `no_control_available` / `thin_stratum` |
| `weights={"income": 3}` | income dominates distance |
| `min_control_pool_size=10` | more thin flags, matching still runs |

Change a value below and re-run the cell.

In [ ]:
strict = MatchConfig(
    match_vars=["income", "age", "region"],
    treatment_col="is_target",
    id_col="id",
    n=1,
    tolerance=0.8,
    min_control_pool_size=5,
    n_bins=4,
)
strict_result = ControlMatcher(strict).fit_match(df)
strict_result.coverage_summary